# Assignment 4 — EvaLoop Telugu-summary loop: run everything

**1. Fill in the settings cell below**, then **Runtime → Run all**.

* `PROVIDER = "vertex"` (default): Gemini on **Vertex AI**, billed to the Google Cloud project that has your course credit.
  Put that project's ID in `PROJECT_ID`. When the sign-in pop-up appears, choose the Google account that owns the project.
  (Optional: instead of signing in, add a Colab secret `VERTEX_API_KEY` with a Vertex AI API key from that project.)
* `PROVIDER = "gemini_api"`: an AI Studio key in a Colab secret `GEMINI_API_KEY` (billing must be on).

Results go to Google Drive (`MyDrive/a4_runs`). If Colab disconnects, just *Run all* again: finished runs are
skipped and every answer already received is replayed from the cache, so nothing is paid for twice.

Generated code is untrusted model output; it is executed only here, inside this Colab VM, by EvalPlus.

In [ ]:
# ---- settings -------------------------------------------------------------
PROVIDER = "vertex"            # "vertex" or "gemini_api"
PROJECT_ID = "tai4se-a4-514318"  # Google Cloud project ID with the course credit (vertex only)
MODELS = "gemini-2.5-flash-lite,gemini-3.1-flash-lite:minimal"

In [ ]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/a4_runs'
os.makedirs(OUT, exist_ok=True)

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

if PROVIDER == "vertex":
    BASE_URL, KEY_ENV = "vertex", "VERTEX_API_KEY"
    os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
    os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
    key = secret("VERTEX_API_KEY")
    if key:
        os.environ["VERTEX_API_KEY"] = key
        print("using Vertex AI API key")
    else:
        os.environ.pop("VERTEX_API_KEY", None)
        assert PROJECT_ID != "your-project-id", "set PROJECT_ID in the settings cell"
        from google.colab import auth
        auth.authenticate_user(project_id=PROJECT_ID)
        print("signed in; project", PROJECT_ID)
else:
    BASE_URL, KEY_ENV = "https://generativelanguage.googleapis.com/v1beta/openai/", "GEMINI_API_KEY"
    os.environ["GEMINI_API_KEY"] = secret("GEMINI_API_KEY") or ""
    assert os.environ["GEMINI_API_KEY"], "add the GEMINI_API_KEY secret"
    print("using Gemini API key")

In [ ]:
%cd /content
!rm -rf EvaLooop
!git clone -q -b a4-telugu-loop https://github.com/IG-08/EvaLooop.git
%cd /content/EvaLooop
!git log --oneline -1
!pip install -q evalplus openai google-genai fire tqdm pytest scipy
!pip install -q -e . --no-deps

## 1. Tests (should end with `103 passed`)

In [ ]:
!python -m pytest -q 2>&1 | tail -3

## 2. Preflight: two real calls per model
Each model should print two `[ok]` lines (one Python function, one Telugu sentence) and a cost estimate.
If you see `[FAIL]`, stop and send a screenshot.

In [ ]:
!python a4/preflight.py --models "$MODELS" --base_url "$BASE_URL" --api_key_env "$KEY_ENV"

## 3. The experiment (about 2–3 hours, mostly waiting)
2 models × 4 configurations on all 378 MBPP+ tasks, temperature 0, max 10 cycles.
Each finished run prints a `[done]` line. Keep this tab open.

In [ ]:
!python a4/run_experiments.py --out "$OUT" --models "$MODELS" --base_url "$BASE_URL" --api_key_env "$KEY_ENV"

## 4. Summary tables and the failed-loop sample

In [ ]:
!python a4/summarize.py --results "$OUT" --out "$OUT/analysis"
!python a4/failures.py --results "$OUT" --out "$OUT/analysis/inspection" --per_cell 8

## 5. Download `a4_results.zip` and attach it in the chat

In [ ]:
import glob
import zipfile
from google.colab import files
zip_path = '/content/a4_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{OUT}/**/*', recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, OUT))
print(round(os.path.getsize(zip_path) / 1e6, 1), 'MB')
files.download(zip_path)